# LLM Container Cold-Start Lab

A reproducible study of where cold-start latency goes when a serverless / scale-to-zero
inference platform brings a model up from nothing: **weight transfer from storage**,
**deserialization into host memory**, **transfer to the accelerator**, and **engine
bring-up** (CUDA context, kernel/JIT warm-up, CUDA-graph capture, KV-cache allocation).

Everything here runs on a **free-tier CPU or a single consumer GPU**, and the results are
extrapolated to production model sizes with an explicit, stated error model.

## How to run
1. Upload `coldstart-lab.zip` (this repo) via the Files pane, **or** run the `git clone` cell.
2. Run the install cell.
3. Pick a GPU runtime (`Runtime -> Change runtime type -> T4`) for the GPU study, or stay on
   CPU for the micro study.
4. Run the experiment cells top to bottom.

## 0. Get the code
Either unzip the uploaded archive, or clone from GitHub. Keep one of these; comment the other.

In [ ]:
# Robust setup: handles nested folders, __MACOSX sidecars and re-uploads.
# Upload coldstart-lab.zip to /content first (Files pane -> Upload).
!unzip -o -q /content/coldstart-lab.zip -d /content
!bash $(find /content -name colab_setup.sh | head -1)

# The setup script prints the project root; cd there so relative paths work.
import subprocess
ROOT = subprocess.check_output(
    "find /content -maxdepth 3 -name pyproject.toml -not -path '*/__MACOSX/*' "
    "-printf '%d %h\\n' | sort -n | head -1 | cut -d' ' -f2-",
    shell=True, text=True).strip()
print('project root:', ROOT)
%cd $ROOT

In [ ]:
# colab_setup.sh already installed the package; this just confirms it.
import coldstart_lab
from coldstart_lab.models import MODEL_REGISTRY
print('coldstart_lab', coldstart_lab.__version__, '-', len(MODEL_REGISTRY), 'models')

## 1. Environment fingerprint
Records exactly what the numbers were measured on. Note whether page-cache control is
available: on an unprivileged runtime we fall back to `posix_fadvise`, which is flagged
in the report.

In [ ]:
from coldstart_lab import environment
fp = environment.probe()
import json; print(json.dumps(fp.to_dict(), indent=2, default=str))
DEVICE = 'cuda' if fp.cuda_available else 'cpu'
print('Using device:', DEVICE)

## 2. The model registry
Organised by weight footprint. `reference` models are **never downloaded** — their
metadata only feeds extrapolation.

In [ ]:
from coldstart_lab.models import MODEL_REGISTRY
for tier in ['ci','micro','small','medium','reference']:
    print(f'\n=== {tier} ===')
    for m in [x for x in MODEL_REGISTRY.values() if x.tier==tier]:
        gate = ' [gated]' if 'gated' in m.tags else ''
        dl = '' if m.downloadable else ' (reference-only)'
        print(f'  {m.key:<20} {m.params_b:>6.2f}B  ~{m.approx_disk_gib:>6.2f} GiB  {m.native_format}{gate}{dl}')

## 3. Recommended run plan

| Runtime | Model to run | Why |
|---|---|---|
| CPU (free) | `smollm2-135m` | Validate the harness end-to-end; format + storage deltas. |
| T4 / L4 | `qwen2.5-3b` | Ungated fp16 model; the core GPU study incl. engine bring-up. |
| T4 / L4 | `qwen2.5-7b` vs `qwen2.5-7b-awq` | Show 4-bit cuts **load** time, not just memory. |
| A100 (if available) | `llama-3.2-3b` | Cross-check throughput on faster storage/PCIe. |

Gated Llama repos need `HF_TOKEN`; set it below if you use them.

In [ ]:
import os
# os.environ['HF_TOKEN'] = 'hf_...'  # only needed for gated (Llama) repos
MODEL = 'smollm2-135m' if DEVICE=='cpu' else 'qwen2.5-3b'
print('Will benchmark:', MODEL, 'on', DEVICE)

## 4. Fetch the checkpoint
The pull itself is a cold-start phase, so we time it.

In [ ]:
from coldstart_lab.fetch import fetch
fetched = fetch(MODEL, token=os.environ.get('HF_TOKEN'))
print(f'pulled in {fetched.pull_ms/1000:.1f}s -> {fetched.local_dir}')
MODEL_DIR = fetched.local_dir

## 5. Experiment A — checkpoint format
safetensors (mmap) vs safetensors (no mmap) vs legacy pickled `.bin`, over identical weights.

In [ ]:
from coldstart_lab.experiments import FormatExperiment
fmt = FormatExperiment(MODEL_DIR, device=DEVICE, include_bin=True, repeats=5, warmup=1)
fmt_res = fmt.run()
import json; print(json.dumps(fmt_res.summary('total_ms'), indent=2))

## 6. Experiment B — storage tier
Local runtime disk vs an emulated bandwidth-capped tier. To use a *real* network tier,
mount Google Drive and add a `StorageTier` rooted under `/content/drive/MyDrive`.

In [ ]:
from coldstart_lab.experiments import StorageExperiment, StorageTier
tiers = [
    StorageTier(name='local-nvme', root='/content/stage'),
    StorageTier(name='remote-emulated-200MiBs', root='/content/stage', emulated_mib_s=200.0),
]
# Real Drive tier (uncomment after mounting):
# from google.colab import drive; drive.mount('/content/drive')
# tiers.append(StorageTier(name='gdrive-nas', root='/content/drive/MyDrive/coldstart'))
storage = StorageExperiment(MODEL_DIR, tiers=tiers, device=DEVICE, repeats=5, warmup=1)
storage_res = storage.run()
print(json.dumps(storage_res.summary('total_ms'), indent=2))

## 7. Experiment C — engine bring-up
On GPU with vLLM installed this compares `enforce_eager` (skip CUDA graphs) vs graph mode.
On CPU it measures the transformers init → weight load → first forward breakdown.

In [ ]:
# For the vLLM comparison on GPU:  !pip install vllm --quiet
from coldstart_lab.experiments import EngineInitExperiment
engine = EngineInitExperiment(MODEL_DIR, device=DEVICE, repeats=3, warmup=0)
print('backends:', engine.backends)
engine_res = engine.run()
print(json.dumps(engine_res.summary('total_ms'), indent=2))

## 8. Extrapolate to production sizes
Take the fastest measured throughput and project weight-load time onto the 32B / 70B class.

In [ ]:
from coldstart_lab.extrapolate import project_load_time
from coldstart_lab.models import models_in_tier
best_tp = max(t.metrics.get('throughput_mib_s', 0.0)
              for r in [fmt_res, storage_res] for t in r.trials)
for p in project_load_time(best_tp, models_in_tier('reference'), basis=f'{best_tp:.0f} MiB/s'):
    print(f"{p.model_key:<16} {p.approx_disk_gib:>6.1f} GiB  ->  {p.predicted_load_s:>6.1f}s predicted load")

## 9. Write the report
Produces a self-contained JSON + Markdown artifact (and PNG charts) — this is the deliverable.

In [ ]:
from coldstart_lab.report import Report, plot_experiment
report = Report(fp)
for r in [fmt_res, storage_res, engine_res]:
    report.add(r)
report.add_extra('pull', {'model': MODEL, 'pull_ms': round(fetched.pull_ms, 1)})
report.add_extra('production_load_projection',
    [p.to_dict() for p in project_load_time(best_tp, models_in_tier('reference'), basis=f'{best_tp:.0f} MiB/s')])
report.write_json(f'/content/{MODEL}_report.json')
report.write_markdown(f'/content/{MODEL}_report.md')
for r in [fmt_res, storage_res, engine_res]:
    plot_experiment(r, f'/content/{MODEL}_{r.name}.png')
print('Report written to /content. Open the .md to read, download the .json to share.')
print(open(f'/content/{MODEL}_report.md').read())